# Google News Explorer

Advancing AI Anthropology through computational approaches to qualitative research.

[Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

<br>

---

<br>

## What This Notebook Does

This notebook searches Google News and exports structured results for qualitative research. Enter search terms, select a time period and country, and download news headlines, sources, dates, and descriptions as CSV or Excel.

The notebook uses the `gnews` library to query Google News RSS feeds — no API key required. Results include article titles, publication dates, source publishers, descriptions, and URLs.

## Key Features

1. **No API Key Required**: Queries Google News directly via public RSS feeds
2. **Keyword Search**: Search for any topic across Google News
3. **Time Period Filtering**: Restrict results to the past hour, day, week, month, or year
4. **Country & Language Selection**: Target news from specific countries and languages
5. **Top News Access**: Retrieve current top headlines without a search query
6. **Structured Export**: Download results as CSV or Excel for qualitative analysis

## Workflow

1. **Configure**: Enter search terms, select time period and country
2. **Fetch**: Retrieve news results from Google News
3. **Review**: Preview headlines and metadata in a table
4. **Export**: Download structured data for further analysis

## Limitations

Google News RSS feeds return a **maximum of 100 results per query**. This is a constraint of the data source, not the notebook. For research requiring larger corpora, consider:

- Running multiple queries with different search terms to broaden coverage
- Using narrower time windows and combining results across periods
- Supplementing with dedicated news databases (LexisNexis, ProQuest, GDELT) for comprehensive media analysis

For most qualitative research tasks — discourse analysis, media framing studies, tracking coverage of specific topics — 100 results per query provides a workable starting point.

## Applications

This notebook supports research that involves analyzing media coverage — tracking how topics are framed in news, collecting public discourse for qualitative coding, identifying patterns in media attention, and contextualizing fieldwork within broader news narratives. The structured exports are formatted for use with qualitative analysis software and other tools in the AI Anthropology Toolkit.

## Methodological Positioning

This notebook represents a **computational anthropology** approach — using programmatic data retrieval to support media analysis in qualitative research. The notebook collects and structures news data but does not interpret it. Analytical judgment remains with the researcher.

**Important**: News articles are copyrighted material. This notebook retrieves metadata (titles, descriptions, dates, sources) and links. Respect publisher terms of service when using this data in research.

## Target Audience

Designed for anthropologists and qualitative researchers who need to collect and analyze news media — from graduate students conducting discourse analysis to research teams tracking media coverage of specific topics.

## Technical Approach

The notebook uses the **gnews** library to query Google News RSS feeds. It supports keyword search, time-based filtering, country/language selection, and top headline retrieval. All processing runs locally in the notebook.

## License & Attribution

This notebook is released under the **CC BY-NC 4.0** license. You may adapt and share it for non-commercial purposes with proper attribution.

## References

Artz, Matt. 2023. From Machine Learning to Machine Knowing: A Digital Anthropology Approach for the Machine Interpretation of Cultures. UNESCO. https://unesdoc.unesco.org/ark:/48223/pf0000384902.

Artz, Matt. 2023. "Ten Predictions for AI and the Future of Anthropology." Anthropology News, May 8. https://doi.org/10.1111/AN.1605.

Artz, Matt. 2026. "Artificial Intelligence: The AI Anthropology Lifecycle (of, by, for AI)." In Practicing Digital Ethnography, edited by Devin Proctor. Routledge. https://doi.org/10.4324/9781032672663-29.

Artz, Matt. 2026. "Multi-Agent Ethnography: Post-Conventional Anthropological Practice Through Human-AI Collaboration." Human Organization. https://doi.org/10.1080/00664677.2026.2614501.

Artz, Matt. Forthcoming. "AI Anthropology: The Future of Applied Anthropological Practice." In Routledge Handbook of Applied Anthropology, edited by Christina Wasson, Edward B. Liebow, Karine L. Narahara, Ndukuyakhe Ndlovu, and Alaka Wali. New York: Routledge.

## Setup and Installation

Install required Python packages and import libraries for Google News retrieval, data processing, and interactive configuration. Run this cell first to ensure all dependencies are available.

In [ ]:
# Install required packages
!pip install gnews pandas openpyxl ipywidgets -q

import pandas as pd
from datetime import datetime
from dateutil import parser as dateparser
import warnings
warnings.filterwarnings('ignore')

from gnews import GNews
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print("\u2713 All packages installed and libraries loaded successfully")
print(f"\u2713 Environment: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print("\U0001f4f0 Ready to configure your Google News search!")

## Search Configuration

Configure your Google News search using the interactive controls below. Set your search terms, time period, country, and export preferences.

In [ ]:
# Interactive Search Interface

style = {'description_width': '120px'}
layout = widgets.Layout(width='500px')

instructions_html = '<div style="background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;">'
instructions_html += '<h2 style="color: #274C77; margin-top: 0;">\U0001f4f0 Google News Explorer</h2>'
instructions_html += '<p><strong>Welcome!</strong> This notebook searches Google News and exports structured results for qualitative research.</p>'
instructions_html += '<h3 style="color: #274C77;">\U0001f4d6 How to Use:</h3>'
instructions_html += '<ol>'
instructions_html += '<li><strong>Search:</strong> Enter keywords or select Top News below</li>'
instructions_html += '<li><strong>Configure:</strong> Set time period, country, and result limit</li>'
instructions_html += '<li><strong>Fetch:</strong> Click the button to retrieve news results</li>'
instructions_html += '<li><strong>Export:</strong> Download as CSV or Excel</li>'
instructions_html += '</ol>'
instructions_html += '<p style="margin-top: 10px; color: #8B8C89; font-size: 0.9em;">'
instructions_html += 'Google News RSS feeds return a maximum of 100 results per query. For larger corpora, run multiple queries with different terms or time windows.</p>'
instructions_html += '</div>'

instructions = widgets.HTML(value=instructions_html)

search_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f50d Search</h3>')

query_input = widgets.Text(
    value='',
    placeholder='Enter search terms (leave empty for Top News)',
    description='Keywords:',
    layout=layout,
    style=style
)

query_help = widgets.HTML(
    '<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">'
    'Enter keywords to search Google News. Leave empty and click Fetch to get current top headlines.</p>'
)

config_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\u2699\ufe0f Filters</h3>')

period_input = widgets.Dropdown(
    options=[
        ('Past hour', '1h'),
        ('Past day', '1d'),
        ('Past week', '7d'),
        ('Past month', '30d'),
        ('Past year', '1y'),
    ],
    value='7d',
    description='Time period:',
    layout=layout,
    style=style
)

country_input = widgets.Dropdown(
    options=[
        ('United States', 'US'),
        ('United Kingdom', 'GB'),
        ('Canada', 'CA'),
        ('Australia', 'AU'),
        ('India', 'IN'),
        ('Germany', 'DE'),
        ('France', 'FR'),
        ('Brazil', 'BR'),
        ('Mexico', 'MX'),
        ('Japan', 'JP'),
    ],
    value='US',
    description='Country:',
    layout=layout,
    style=style
)

lang_input = widgets.Dropdown(
    options=[
        ('English', 'en'),
        ('Spanish', 'es'),
        ('French', 'fr'),
        ('German', 'de'),
        ('Portuguese', 'pt'),
        ('Japanese', 'ja'),
    ],
    value='en',
    description='Language:',
    layout=layout,
    style=style
)

max_results_input = widgets.IntSlider(
    value=50, min=10, max=100, step=10,
    description='Max results:',
    style=style
)

max_help = widgets.HTML(
    '<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">'
    'Google News returns up to 100 results per query. Actual results may be fewer depending on the topic and time period.</p>'
)

export_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f4c1 Export</h3>')

format_input = widgets.Dropdown(
    options=[('CSV (.csv)', 'csv'), ('Excel (.xlsx)', 'xlsx')],
    value='csv',
    description='Format:',
    layout=layout,
    style=style
)

fetch_btn = widgets.Button(
    description='Fetch News',
    button_style='',
    layout=widgets.Layout(width='200px', margin='20px 0'),
    style={'button_color': '#6096BA'}
)

out = widgets.Output()


def on_fetch_clicked(_):
    out.clear_output()
    with out:
        query = query_input.value.strip()
        is_top_news = not query

        gn = GNews(
            language=lang_input.value,
            country=country_input.value,
            period=period_input.value,
            max_results=max_results_input.value,
        )

        if is_top_news:
            print("\U0001f4f0 Fetching top headlines...")
        else:
            print(f"\U0001f50d Searching Google News for: {query}")
        print(f"   Country: {country_input.value}, Language: {lang_input.value}")
        print(f"   Period: {period_input.value}, Max results: {max_results_input.value}")
        print()

        try:
            if is_top_news:
                results = gn.get_top_news()
            else:
                results = gn.get_news(query)

            if not results:
                print("\u26a0\ufe0f No results found. Try different search terms or a broader time period.")
                return

            print(f"\u2713 Retrieved {len(results)} articles")
            print()

            # Build DataFrame
            rows = []
            for r in results:
                pub = r.get('publisher', {})
                rows.append({
                    'title': r.get('title', ''),
                    'source': pub.get('title', '') if isinstance(pub, dict) else str(pub),
                    'published': r.get('published date', ''),
                    'description': r.get('description', ''),
                    'url': r.get('url', ''),
                })

            df = pd.DataFrame(rows)

            # Parse dates for sorting
            try:
                df['date_parsed'] = pd.to_datetime(df['published'], format='mixed', utc=True)
                df = df.sort_values('date_parsed', ascending=False).drop(columns=['date_parsed'])
            except Exception:
                pass

            print("\U0001f4ca Preview:")
            display(df[['title', 'source', 'published']].head(15))

            # Export
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            slug = query.replace(' ', '_')[:30] if query else 'top_news'
            base = f'google_news_{slug}_{timestamp}'
            fmt = format_input.value

            if fmt == 'xlsx':
                filepath = f'{base}.xlsx'
                df.to_excel(filepath, sheet_name='News Results', index=False, engine='openpyxl')

                from openpyxl import load_workbook
                from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
                wb = load_workbook(filepath)
                header_fill = PatternFill(start_color='274C77', end_color='274C77', fill_type='solid')
                header_font = Font(bold=True, color='FFFFFF')
                thin_border = Border(
                    left=Side(style='thin', color='E7ECEF'), right=Side(style='thin', color='E7ECEF'),
                    top=Side(style='thin', color='E7ECEF'), bottom=Side(style='thin', color='E7ECEF'),
                )
                for ws in wb.worksheets:
                    for cell in ws[1]:
                        cell.fill = header_fill
                        cell.font = header_font
                        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
                        cell.border = thin_border
                    for col in ws.columns:
                        max_len = max(len(str(c.value or '')) for c in col)
                        ws.column_dimensions[col[0].column_letter].width = min(max(max_len + 2, 10), 60)
                    ws.freeze_panes = 'A2'
                    for row in ws.iter_rows(min_row=2):
                        for cell in row:
                            cell.alignment = Alignment(vertical='top', wrap_text=True)
                            cell.border = thin_border
                wb.save(filepath)
            else:
                filepath = f'{base}.csv'
                df.to_csv(filepath, index=False)

            print()
            print(f"\u2713 Saved: {filepath} ({len(df)} articles)")

            if IN_COLAB:
                colab_files.download(filepath)
                print(f"\U0001f4e5 Downloaded: {filepath}")

        except Exception as e:
            print(f"\u274c Error: {type(e).__name__}: {e}")


fetch_btn.on_click(on_fetch_clicked)

display(widgets.VBox([
    instructions,
    search_header,
    query_input,
    query_help,
    config_header,
    period_input,
    country_input,
    lang_input,
    max_results_input,
    max_help,
    export_header,
    format_input,
    fetch_btn,
    out,
]))